# 神经网络基础练习参考答案（第 3 章）

本文件是配套练习题的参考答案，请先在手写练习后再对照。

# 神经网络基础练习（第 3 章）

本练习覆盖《尚硅谷大模型技术之深度学习》第 3 章神经网络基础的以下内容：

- 神经网络的构成
- 全连接层（nn.Linear）
- 激活函数
- 搭建神经网络
- 加载神经网络模型
- 应用案例：手写数字识别

使用方法：阅读每道题的描述后，在下方代码单元格中**手写代码**完成练习；写完后可与 `answer` 目录下的答案对照。

部分练习所需的数据位于项目根目录的 `data/` 下，本 notebook 中使用相对路径 `../../data/...`。

先执行下面的单元格导入所需的库。

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

print("torch version:", torch.__version__)

## 3.1 神经网络的构成

**练习 1**：用张量模拟一个神经元

给定输入 `x = [1.0, 2.0, 3.0]`，权重 `w = [0.5, -0.2, 0.1]`，偏置 `b = 0.3`：
1. 用元素级乘法加求和的方式计算加权总和 `a = w1*x1 + w2*x2 + w3*x3 + b`
2. 再用矩阵乘法（`@`）计算一遍，验证两种方式结果一致

打印计算结果。

In [ ]:
# 练习 1：用张量模拟一个神经元

# 练习 1：用张量模拟一个神经元

x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, -0.2, 0.1])
b = torch.tensor(0.3)

a1 = (w * x).sum() + b
print("元素级乘法求和:", a1)

a2 = w @ x + b
print("矩阵乘法     :", a2)

print("是否一致:", torch.allclose(a1, a2))

**练习 2**：三层网络的逐层前向传播

一个三层神经网络：输入层 2 个神经元、第 1 层 3 个、第 2 层 2 个。给定：
- `X` 形状 `(2,)`
- `W1` 形状 `(2, 3)`、`B1` 形状 `(3,)`
- `W2` 形状 `(3, 2)`、`B2` 形状 `(2,)`

请用 `torch.manual_seed(42)` 固定随机种子后随机初始化这些张量，然后：
1. 计算第 1 层：`A1 = X @ W1 + B1`，再经过激活函数 Tanh 得到 `Z1 = tanh(A1)`
2. 计算第 2 层：`A2 = Z1 @ W2 + B2`

打印每一步结果的形状。

In [ ]:
# 练习 2：三层网络的逐层前向传播

# 练习 2：三层网络的逐层前向传播

torch.manual_seed(42)

X = torch.randn(2)          # 输入 (2,)
W1 = torch.randn(2, 3)      # 输入层 -> 第1层
B1 = torch.randn(3)
W2 = torch.randn(3, 2)      # 第1层 -> 第2层
B2 = torch.randn(2)

A1 = X @ W1 + B1
print("A1:", A1, A1.shape)

Z1 = torch.tanh(A1)
print("Z1:", Z1, Z1.shape)

A2 = Z1 @ W2 + B2
print("A2:", A2, A2.shape)

## 3.2 全连接层

**练习 3**：创建全连接层并查看参数

1. 使用 `nn.Linear(2, 3)` 创建一个全连接层
2. 打印 `weight` 与 `bias` 的形状，观察 `weight` 的形状是 `(out_features, in_features)` 还是 `(in_features, out_features)`
3. 用 `named_parameters()` 打印参数名与形状

In [ ]:
# 练习 3：创建全连接层并查看参数

# 练习 3：创建全连接层并查看参数

linear = nn.Linear(2, 3)

print("weight:", linear.weight.shape)
print("bias  :", linear.bias.shape)

for name, param in linear.named_parameters():
    print(name, param.shape)

**练习 4**：全连接层的前向传播

1. 创建一个 `nn.Linear(2, 3)`
2. 输入 `x = torch.randn(5, 2)`，分别用 `linear.forward(x)` 与 `linear(x)` 计算输出，验证两者结果一致
3. 打印输入与输出的形状
（说明：`nn` 中所有模块的 `__call__` 方法都指向自身的 `forward` 方法）

In [ ]:
# 练习 4：全连接层的前向传播

# 练习 4：全连接层的前向传播

linear = nn.Linear(2, 3)
x = torch.randn(5, 2)

y1 = linear.forward(x)
y2 = linear(x)

print("输入:", x.shape)
print("输出:", y1.shape)
print("是否一致:", torch.allclose(y1, y2))

**练习 5**：手写实现全连接层

不使用 `nn.Linear`，用 `nn.Parameter` 定义权重 `weight`（形状 `(out_features, in_features)`）与偏置 `bias`，实现全连接层：

```
y = x @ weight.T + bias
```

要求：
1. 创建 `in_features=2`、`out_features=3` 的可训练参数
2. 将参数拷贝到同形状的 `nn.Linear` 中，验证两者对同一输入 `x = torch.randn(5, 2)` 的输出一致

In [ ]:
# 练习 5：手写实现全连接层

# 练习 5：手写实现全连接层

torch.manual_seed(42)
in_features, out_features = 2, 3

weight = nn.Parameter(torch.randn(out_features, in_features))
bias = nn.Parameter(torch.randn(out_features))

x = torch.randn(5, 2)
y_manual = x @ weight.T + bias
print("手写实现输出:", y_manual.shape)

linear = nn.Linear(in_features, out_features)
with torch.no_grad():
    linear.weight.copy_(weight)
    linear.bias.copy_(bias)

print("是否与 nn.Linear 一致:", torch.allclose(y_manual, linear(x)))

## 3.3 激活函数

**练习 6**：Sigmoid 函数

对 `x = torch.tensor([-6., -3., -1., 0., 1., 3., 6.])`：
1. 分别用 `torch.sigmoid(x)`、`x.sigmoid()`、`nn.Sigmoid()(x)` 三种方式计算，验证结果一致
2. 验证输出都落在 `(0, 1)` 区间
3. 按导数公式 `f'(x) = f(x) * (1 - f(x))` 计算导数，观察输入在 `[-6, 6]` 之外时导数接近 0（梯度消失）

In [ ]:
# 练习 6：Sigmoid 函数

# 练习 6：Sigmoid 函数

x = torch.tensor([-6., -3., -1., 0., 1., 3., 6.])

y1 = torch.sigmoid(x)
y2 = x.sigmoid()
y3 = nn.Sigmoid()(x)
print("torch.sigmoid:", y1)
print("三种方式结果一致:", torch.allclose(y1, y2) and torch.allclose(y1, y3))

print("输出是否都在 (0, 1) 内:", bool(((y1 > 0) & (y1 < 1)).all()))

grad = y1 * (1 - y1)
print("导数 f'(x)=f(x)(1-f(x)):", grad)

**练习 7**：Tanh 函数

对 `x = torch.linspace(-6, 6, 13)`：
1. 用 `torch.tanh(x)` 计算，验证输出范围在 `(-1, 1)` 内
2. 验证 Tanh 关于原点中心对称：`torch.tanh(-x) == -torch.tanh(x)`
3. 用导数公式 `f'(x) = 1 - f(x)^2` 计算导数

In [ ]:
# 练习 7：Tanh 函数

# 练习 7：Tanh 函数

x = torch.linspace(-6, 6, 13)

y = torch.tanh(x)
print("tanh:", y)
print("输出是否都在 (-1, 1) 内:", bool(((y > -1) & (y < 1)).all()))
print("是否关于原点中心对称:", torch.allclose(torch.tanh(-x), -y))

grad = 1 - y ** 2
print("导数 f'(x)=1-f(x)^2:", grad)

**练习 8**：ReLU 函数

1. 对 `x = torch.tensor([-2., -1., 0., 1., 2.])` 用 `torch.relu(x)` 计算
2. 手动用 `torch.maximum(x, torch.zeros_like(x))` 实现一遍，验证结果一致
3. 用 `nn.ReLU()` 模块实现一遍，验证结果一致

In [ ]:
# 练习 8：ReLU 函数

# 练习 8：ReLU 函数

x = torch.tensor([-2., -1., 0., 1., 2.])

y1 = torch.relu(x)
y2 = torch.maximum(x, torch.zeros_like(x))
y3 = nn.ReLU()(x)

print("torch.relu       :", y1)
print("手动 max(0, x)    :", y2)
print("nn.ReLU          :", y3)
print("三种方式一致:", torch.allclose(y1, y2) and torch.allclose(y1, y3))

**练习 9**：Leaky ReLU 与 PReLU

对含负数的输入 `x = torch.tensor([-3., -1., 0., 1., 3.])`：
1. 分别用 `nn.LeakyReLU(0.1)` 与 `nn.PReLU()` 计算输出
2. 打印 `nn.PReLU()` 的可训练参数，说明它与 Leaky ReLU 的区别

In [ ]:
# 练习 9：Leaky ReLU 与 PReLU

# 练习 9：Leaky ReLU 与 PReLU

x = torch.tensor([-3., -1., 0., 1., 3.])

leaky = nn.LeakyReLU(0.1)
prelu = nn.PReLU()

print("LeakyReLU(0.1):", leaky(x))
print("PReLU         :", prelu(x))

print("PReLU 的可训练参数:")
for name, param in prelu.named_parameters():
    print(" ", name, param.data)
# Leaky ReLU 的负半轴斜率 0.1 是固定常数；PReLU 的斜率是可训练参数

**练习 10**：Softmax 函数

1. 对 `x = torch.tensor([1., 3., 5., 10.])` 用 `torch.softmax(x, dim=0)` 计算，验证结果之和为 1
2. 对 `torch.randn(3, 4)` 用 `torch.softmax(x, dim=1)` 计算，验证每一行的和都为 1
3. 用 `nn.Softmax(dim=1)` 模块再算一遍二维张量，验证与第 2 步一致

In [ ]:
# 练习 10：Softmax 函数

# 练习 10：Softmax 函数

x = torch.tensor([1., 3., 5., 10.])
y = torch.softmax(x, dim=0)
print("一维 softmax:", y)
print("和为:", y.sum().item())

torch.manual_seed(42)
x2 = torch.randn(3, 4)
y2 = torch.softmax(x2, dim=1)
print("二维 softmax 每行和:", y2.sum(dim=1))

y3 = nn.Softmax(dim=1)(x2)
print("nn.Softmax 与 torch.softmax 一致:", torch.allclose(y2, y3))

**练习 11**：其他常见激活函数

对同一个输入 `x = torch.linspace(-3, 3, 7)`，分别用以下模块计算输出并打印：
- `nn.Identity()`（恒等函数）
- `nn.ELU()`
- `nn.SiLU()`（即 Swish）
- `nn.Softplus()`

In [ ]:
# 练习 11：其他常见激活函数

# 练习 11：其他常见激活函数

x = torch.linspace(-3, 3, 7)

ident = nn.Identity()
elu = nn.ELU()
silu = nn.SiLU()
softplus = nn.Softplus()

print("Identity:", ident(x))
print("ELU     :", elu(x))
print("SiLU    :", silu(x))
print("Softplus:", softplus(x))

## 3.4 搭建神经网络

**练习 12**：自定义 nn.Module 搭建三层神经网络

按下面的结构实现模型（输入 3 维 → 隐藏层 4 → 隐藏层 4 → 输出 2 维）：
- 第 1 个隐藏层：`nn.Linear(3, 4)`，激活函数 Tanh
- 第 2 个隐藏层：`nn.Linear(4, 4)`，激活函数 ReLU
- 输出层：`nn.Linear(4, 2)`，激活函数 Softmax（`dim=1`）

要求：
1. 继承 `nn.Module`，实现 `__init__` 与 `forward` 两个方法
2. 输入 `torch.randn(10, 3)` 做前向传播，打印输出
3. 验证输出每一行的和都为 1

In [ ]:
# 练习 12：自定义 nn.Module 搭建三层神经网络

# 练习 12：自定义 nn.Module 搭建三层神经网络

class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.linear1 = nn.Linear(3, 4)   # 第1个隐藏层：3 输入，4 输出
        self.linear2 = nn.Linear(4, 4)   # 第2个隐藏层：4 输入，4 输出
        self.out = nn.Linear(4, 2)       # 输出层：4 输入，2 输出

    def forward(self, x):
        x = self.linear1(x)
        x = torch.tanh(x)                # 第1个隐藏层激活函数
        x = self.linear2(x)
        x = torch.relu(x)                # 第2个隐藏层激活函数
        x = self.out(x)
        x = torch.softmax(x, dim=1)      # 输出层激活函数
        return x

torch.manual_seed(42)
model = Model()
x = torch.randn(10, 3)
output = model(x)
print("输出：\n", output)
print("每行之和:", output.sum(dim=1))

**练习 13**：查看模型结构与参数数量

基于练习 12 的模型：
1. 用 `named_parameters()` 打印各层参数名与形状
2. 用 `state_dict()` 打印状态字典
3. 统计模型的总参数量，并与手算结果核对（每个 `Linear(in, out)` 的参数个数为 `in * out + out`）

In [ ]:
# 练习 13：查看模型结构与参数数量

# 练习 13：查看模型结构与参数数量

torch.manual_seed(42)
model = Model()

print("各层参数：")
for name, param in model.named_parameters():
    print(name, param.shape)

print("\nstate_dict 键：", list(model.state_dict().keys()))

total = sum(p.numel() for p in model.parameters())
print("\n总参数量:", total)
# 手算：(3*4+4) + (4*4+4) + (4*2+2) = 16 + 20 + 10 = 46
print("手算参数量:", 3 * 4 + 4 + 4 * 4 + 4 + 4 * 2 + 2)

**练习 14**：使用 nn.Sequential 构建模型

用 `nn.Sequential` 构建与练习 12 相同结构的模型（`Linear(3,4)` → `Tanh` → `Linear(4,4)` → `ReLU` → `Linear(4,2)` → `Softmax(dim=1)`），
输入 `torch.randn(10, 3)` 做前向传播并打印输出。

In [ ]:
# 练习 14：使用 nn.Sequential 构建模型

# 练习 14：使用 nn.Sequential 构建模型

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.Tanh(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
    nn.Softmax(dim=1),
)

output = model(torch.randn(10, 3))
print("输出：\n", output)

## 3.5 加载神经网络模型

**练习 15**：保存与加载模型参数

1. 用 `nn.Sequential` 新建一个模型，用 `torch.save(model.state_dict(), "model.pt")` 把参数保存到文件
2. 新建一个同结构的模型，用 `torch.load("model.pt")` 读取参数并 `load_state_dict()` 加载
3. 对同一输入 `torch.randn(10, 3)` 验证两个模型的输出是否一致

In [ ]:
# 练习 15：保存与加载模型参数

# 练习 15：保存与加载模型参数

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.Tanh(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
    nn.Softmax(dim=1),
)

# 1. 保存模型参数（状态字典）
torch.save(model.state_dict(), "model.pt")
print("已保存到 model.pt")

# 2. 新建同结构模型并加载参数
model2 = nn.Sequential(
    nn.Linear(3, 4),
    nn.Tanh(),
    nn.Linear(4, 4),
    nn.ReLU(),
    nn.Linear(4, 2),
    nn.Softmax(dim=1),
)
state_dict = torch.load("model.pt")
model2.load_state_dict(state_dict)

# 3. 验证输出一致
x = torch.randn(10, 3)
print("加载前后输出一致:", torch.allclose(model(x), model2(x)))

## 3.6 应用案例：手写数字识别

**练习 16**：编写数据读取函数 `get_data()`

读取 `../../data/train.csv`（Digit Recognizer 数据集，第 1 列为标签，之后 784 列为 28×28 像素亮度值），
编写函数 `get_data()` 返回 `x_train, x_test, y_train, y_test`：

1. 用 `pd.read_csv` 加载数据集
2. `X = data.drop("label", axis=1)` 作为特征，`y = data["label"]` 作为标签
3. 用 `train_test_split` 按 `test_size=0.3, random_state=42` 划分训练集与测试集
4. 用 `MinMaxScaler` 做归一化（训练集 `fit_transform`，测试集 `transform`）
5. 转成 `float` 类型的张量后返回

调用该函数并打印各个变量的形状。

In [ ]:
# 练习 16：编写数据读取函数 get_data()

# 练习 16：编写数据读取函数 get_data()

from sklearn.model_selection import train_test_split   # 划分数据集
from sklearn.preprocessing import MinMaxScaler         # 归一化 Scaler


def get_data():
    # 1. 加载数据集
    data = pd.read_csv("../../data/train.csv")
    # 2. 划分训练集和测试集
    X = data.drop("label", axis=1)   # 特征
    y = data["label"]                # 标签
    x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    # 3. 特征转换：归一化
    scaler = MinMaxScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    # 4. 转成 tensor
    x_train = torch.tensor(x_train).float()
    x_test = torch.tensor(x_test).float()
    y_train = torch.tensor(y_train.values)
    y_test = torch.tensor(y_test.values)
    return x_train, x_test, y_train, y_test


x_train, x_test, y_train, y_test = get_data()
print("x_train:", x_train.shape, x_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("y_test :", y_test.shape, y_test.dtype)

**练习 17**：加载模型进行手写数字识别推理

1. 用 `nn.Sequential` 构建三层神经网络：输入层 784 个神经元，第 1 个隐藏层 50 个、第 2 个隐藏层 100 个（激活函数均为 ReLU），输出层 10 个神经元（代表 0~9）
2. 用 `torch.load("../../data/nn_example.pt")` 加载已经训练好的模型参数（该文件保存于 GPU，若本机没有 GPU，需加上 `map_location="cpu"`）
3. 按 `batch_size=100` 分批对测试集做前向传播，用 `torch.argmax(y, dim=1)` 得到预测结果
4. 统计并打印模型在测试集上的准确率

In [ ]:
# 练习 17：加载模型进行手写数字识别推理

# 练习 17：加载模型进行手写数字识别推理

# 1. 读入数据（直接复用练习 16 中定义的 get_data 函数；也可将其保存为 load_data.py 后导入）
x_train, x_test, y_train, y_test = get_data()

# 2. 创建模型
model = nn.Sequential(
    nn.Linear(784, 50),
    nn.ReLU(),
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 10),
)

# 3. 加载模型参数（该文件保存于 GPU，加载到 CPU 需指定 map_location）
state_dict = torch.load("../../data/nn_example.pt", map_location="cpu")
model.load_state_dict(state_dict)

batch_size = 100        # 批数量
n = x_test.shape[0]     # 数据个数
accuracy_cnt = 0        # 累计准确数量
for i in range(0, n, batch_size):
    # 1. 取出当前批次的测试数据
    x_batch = x_test[i: i + batch_size]
    y_batch = y_test[i: i + batch_size]
    # 2. 前向传播
    y = model(x_batch)
    # 3. 得到预测结果
    y_pred = torch.argmax(y, dim=1)
    # 4. 计算准确数量
    accuracy_cnt += y_pred.eq(y_batch).sum().item()

# 打印准确率
print("Accuracy:", accuracy_cnt / n)